In [11]:
import os, shutil, json, re
from pathlib import Path
import pandas as pd
from datetime import datetime
from collections import Counter

# Define BIDS root and derivatives roots
dataset = "bruxelles-bids"
bids_root = Path(f"/data/project/sleep_ENIGMA_glymphatic/enigma-sd/{dataset}")
derivatives_roots = [bids_root / "derivatives/fmriprep", bids_root / "derivatives/freesurfer"]

# Read participant data
participants = pd.read_csv(bids_root / "participants.tsv", sep="\t")
crossover = "first_session" in participants.columns  # is the data a crossover design?

# Session label maps
session_map = {
    "ses-T2BL": "ses-T1BL",
}
crossover_map = {  # used when first_session == "SR"
    "ses-T1BL": "ses-T2SD",
    "ses-T2BL": "ses-T1BL"
}
control_map = {  # used when subject is a control (group == 1)
    "ses-T1BL": "ses-T2SD",
    "ses-T2BL": "ses-T1BL"
}

def build_session_map(first_session=None, group=None):
    is_control = pd.notna(group) and int(group) < 3

    if is_control:
        return control_map
    elif not crossover or pd.isna(first_session) or first_session == "BL":
        return session_map
    elif first_session == "SR":
        return crossover_map
    else:
        raise ValueError(f"Unexpected first_session value: {first_session}")

subject_session_maps = {
    row["participant_id"]: build_session_map(
        first_session=row["first_session"] if crossover else None,
        group=row["group"]
    )
    for _, row in participants.iterrows()
}

if not crossover:
    print("No 'first_session' column found — applying session_map to all subjects.")

print(f"Built session maps for {len(subject_session_maps)} subjects. Sample of 5:")
for k in list(subject_session_maps)[:5]:
    print(k, subject_session_maps[k])

# Identify conditions
condition_map = {
    "CL": "control",
    "BL": "baseline",
    "SD": "sleep deprivation",
    "SR": "sleep restriction",
    "RC": "recovery sleep"
}

No 'first_session' column found — applying session_map to all subjects.
Built session maps for 31 subjects. Sample of 5:
sub-001 {'ses-T2BL': 'ses-T1BL'}
sub-002 {'ses-T2BL': 'ses-T1BL'}
sub-003 {'ses-T2BL': 'ses-T1BL'}
sub-004 {'ses-T2BL': 'ses-T1BL'}
sub-005 {'ses-T2BL': 'ses-T1BL'}


Scan the dataset for old session labels:

In [12]:
# Collect all items with any old session label
old_labels = set(session_map.keys())

all_items = []
label_counter = Counter()
for item in bids_root.rglob("*"):
    if "sourcedata" in item.parts: # Ignore sourcedata folder
        continue
    item_str = str(item)
    matched = [old_ses for old_ses in old_labels if old_ses in item_str]
    if matched:
        all_items.append(item)
        label_counter.update(matched)

print(f"Found {len(all_items)} items containing old session labels.")
print("Occurrences of old session labels:")
for label, count in label_counter.items():
    print(f"{label}: {count} items")

Found 341 items containing old session labels.
Occurrences of old session labels:
ses-T2BL: 341 items


Give a sample of files to be renamed:

In [13]:
# Helper function for matching subject IDs
def get_subject_id(path):
    match = re.search(r"(sub-[a-zA-Z0-9]+)", str(path))
    return match.group(1) if match else None

preview_renames = []
no_subject_generic = []

for item in all_items:
    subj = get_subject_id(item)
    if subj is not None and subj in subject_session_maps:
        ses_map = subject_session_maps[subj]
    else:
        ses_map = session_map
        no_subject_generic.append(item)

    new_path = Path(str(item))
    for old_ses, new_ses in ses_map.items():
        new_path = Path(str(new_path).replace(old_ses, new_ses))
    preview_renames.append((item, new_path))

# Preview first 10
print("Sample of renamed files/folders (first 10):")
for old, new in preview_renames[:10]:
    print(f"{old} -> {new}")

if no_subject_generic:
    print(f"\n{len(no_subject_generic)} items had no subject match — used generic session_map:")
    generic_renames = {old: new for old, new in preview_renames if old in no_subject_generic}
    for s in no_subject_generic[:10]:
        print(f"{s} -> {generic_renames[s]}")

Sample of renamed files/folders (first 10):
/data/project/sleep_ENIGMA_glymphatic/enigma-sd/bruxelles-bids/sub-008/ses-T2BL -> /data/project/sleep_ENIGMA_glymphatic/enigma-sd/bruxelles-bids/sub-008/ses-T1BL
/data/project/sleep_ENIGMA_glymphatic/enigma-sd/bruxelles-bids/sub-008/ses-T2BL/dwi -> /data/project/sleep_ENIGMA_glymphatic/enigma-sd/bruxelles-bids/sub-008/ses-T1BL/dwi
/data/project/sleep_ENIGMA_glymphatic/enigma-sd/bruxelles-bids/sub-008/ses-T2BL/anat -> /data/project/sleep_ENIGMA_glymphatic/enigma-sd/bruxelles-bids/sub-008/ses-T1BL/anat
/data/project/sleep_ENIGMA_glymphatic/enigma-sd/bruxelles-bids/sub-008/ses-T2BL/dwi/sub-008_ses-T2BL_dir-AP_dwi.json -> /data/project/sleep_ENIGMA_glymphatic/enigma-sd/bruxelles-bids/sub-008/ses-T1BL/dwi/sub-008_ses-T1BL_dir-AP_dwi.json
/data/project/sleep_ENIGMA_glymphatic/enigma-sd/bruxelles-bids/sub-008/ses-T2BL/dwi/sub-008_ses-T2BL_dir-AP_dwi.nii.gz -> /data/project/sleep_ENIGMA_glymphatic/enigma-sd/bruxelles-bids/sub-008/ses-T1BL/dwi/sub-00

Rename files:

In [14]:
# Rename files first
renamed_count = 0
errors = []
for old_path, new_path in preview_renames:
    if old_path.is_file():
        try:
            new_path.parent.mkdir(parents=True, exist_ok=True)
            old_path.rename(new_path)
            renamed_count += 1
        except Exception as e:
            errors.append((old_path, new_path, str(e)))
print(f"Renamed {renamed_count} files.")
if errors:
    print(f"{len(errors)} errors occurred:")
    for old, new, err in errors[:10]:
        print(f"{old} -> {new}: {err}")

Renamed 248 files.


Then rename directories using merging:

In [15]:
def safe_rename_dir(src, dst):
    if not dst.exists():
        src.rename(dst)
    else:
        # Merge contents
        for item in src.iterdir():
            target = dst / item.name
            if item.is_dir():
                safe_rename_dir(item, target)
            else:
                if target.exists():
                    # If file exists, overwrite
                    target.unlink()
                shutil.move(str(item), str(target))
        # Remove the now-empty source directory
        src.rmdir()

# Collect directories containing any old session label (excluding sourcedata)
all_dirs = [
    d for d in bids_root.rglob("*")
    if d.is_dir()
    and "sourcedata" not in d.parts
    and any(old in str(d) for old in old_labels)
]

# Deepest first 
all_dirs = sorted(all_dirs, key=lambda x: len(str(x)), reverse=True)

renamed_dir_count = 0
dir_errors = []
for dir_path in all_dirs:
    subj = get_subject_id(dir_path)
    if subj is not None and subj in subject_session_maps:
        ses_map = subject_session_maps[subj]
    else:
        ses_map = session_map

    new_dir_path = Path(str(dir_path))
    for old_ses, new_ses in ses_map.items():
        new_dir_path = Path(str(new_dir_path).replace(old_ses, new_ses))

    try:
        new_dir_path.parent.mkdir(parents=True, exist_ok=True)
        safe_rename_dir(dir_path, new_dir_path)
        renamed_dir_count += 1
    except Exception as e:
        dir_errors.append((dir_path, new_dir_path, str(e)))

print(f"Renamed {renamed_dir_count} directories (out of {len(all_dirs)} found).")
if dir_errors:
    print(f"{len(dir_errors)} errors occurred:")
    for old, new, err in dir_errors[:10]:
        print(f"{old} -> {new}: {err}")

Renamed 93 directories (out of 93 found).


In [16]:
def infer_condition(session_label):
    for key, val in condition_map.items():
        if key.lower() in session_label.lower():
            return val
    return "unknown"

def extract_visit(session_label):
    match = re.search(r'(T\d+)', session_label, re.IGNORECASE)
    if match:
        return match.group(1).upper()
    return "n/a"

def extract_date_time(json_file):
    date, time = "n/a", "n/a"
    try:
        with open(json_file, "r") as f:
            meta = json.load(f)
        raw_date = meta.get("SeriesDate") or meta.get("AcquisitionDate") or meta.get("StudyDate")
        if raw_date:
            raw_date = str(raw_date)
            if "-" in raw_date:
                date = raw_date
            else:
                date = f"{raw_date[:4]}-{raw_date[4:6]}-{raw_date[6:8]}"
        raw_time = meta.get("SeriesTime") or meta.get("AcquisitionTime") or meta.get("StudyTime")
        if raw_time:
            raw_time = str(raw_time).split(".")[0]
            if ":" in raw_time:
                time = raw_time
            else:
                time = f"{raw_time[:2]}:{raw_time[2:4]}:{raw_time[4:6]}"
    except Exception:
        pass
    return date, time

# Iterate over subjects
subjects = sorted([p for p in bids_root.glob("sub-*") if p.is_dir()])
out_files = []

for sub in subjects:
    participant_id = sub.name
    sessions = sorted([s for s in sub.glob("ses-*") if s.is_dir()])
    rows = []
    for ses in sessions:
        session_id = ses.name
        visit = extract_visit(session_id)
        anat_jsons = list((ses / "anat").glob("*T1w.json")) if (ses / "anat").exists() else []
        json_file = anat_jsons[0] if anat_jsons else None
        date, time = extract_date_time(json_file) if json_file else ("n/a", "n/a")
        rows.append({
            "participant_id": participant_id,
            "session_id": session_id,
            "dicom_id": "n/a",
            "visit": visit,
            "condition": infer_condition(session_id),
            "date": date,
            "time": time
        })
    df = pd.DataFrame(rows)
    out_file = sub / f"{participant_id}_sessions.tsv"

    # Unlock if it already exists (annexed symlink) before writing
    if out_file.exists():
        !datalad unlock "{out_file}"

    df.to_csv(out_file, sep="\t", index=False)
    out_files.append(str(out_file))
    print(f"Saved: {out_file.name}")

print(f"\nWrote {len(out_files)} sessions.tsv files. Committing batch...")

unlock(ok): bruxelles-bids/sub-001/sub-001_sessions.tsv (file)
                                                                            Saved: sub-001_sessions.tsv
unlock(ok): bruxelles-bids/sub-002/sub-002_sessions.tsv (file)
                                                                            Saved: sub-002_sessions.tsv
unlock(ok): bruxelles-bids/sub-003/sub-003_sessions.tsv (file)
                                                                            Saved: sub-003_sessions.tsv
unlock(ok): bruxelles-bids/sub-004/sub-004_sessions.tsv (file)
                                                                            Saved: sub-004_sessions.tsv
unlock(ok): bruxelles-bids/sub-005/sub-005_sessions.tsv (file)
                                                                            Saved: sub-005_sessions.tsv
unlock(ok): bruxelles-bids/sub-006/sub-006_sessions.tsv (file)
                                                                            Saved: sub-006_sessions.ts

Sanity check:

In [18]:
df

,participant_id,session_id,dicom_id,visit,condition,date,time
0,sub-33,ses-T1BL,n/a,T1,baseline,n/a,09:27:2
1,sub-33,ses-T2SD,n/a,T2,sleep deprivation,n/a,10:42:42
